In [432]:
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.auto import tqdm

workspace_root = None
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "data" / "meteorology_dataset.csv").exists() and (candidate / "level2").exists():
        workspace_root = candidate
        break

if workspace_root is None:
    workspace_root = Path.cwd()

saved_models_dir = workspace_root / "level2" / "saved_models"
saved_models_dir.mkdir(parents=True, exist_ok=True)

def save_model_artifact(model_name, model, imputer, feature_columns, metadata=None):
    artifact_path = saved_models_dir / f"{model_name}.joblib"
    artifact = {
        "model": model,
        "imputer": imputer,
        "feature_columns": list(feature_columns),
        "excluded_features": list(excluded_features),
        "metadata": metadata or {},
    }
    dump(artifact, artifact_path)
    print(f"Saved {model_name} to {artifact_path}")
    return artifact_path

df = pd.read_csv("../data/meteorology_dataset.csv")
df_name = "meteorology_dataset.csv"
data = df.copy()

# Normalize headers once to avoid hidden-space mismatches.
data.columns = [str(col).strip() for col in data.columns]

time_col = "time"
temp_col = "temperature_2m"
dew_col = "dew_point_2m"
humidity_col = "relative_humidity_2m"

# User-controlled exclusions from the final training feature set.
excluded_features = ["dayofweek", "day", "wind_speed_10m", "time", "year"]

if time_col in data.columns:
    data[time_col] = pd.to_datetime(data[time_col], errors="coerce")
    data = data.sort_values(time_col).reset_index(drop=True)

if temp_col not in data.columns:
    raise KeyError(
        f"Could not find '{temp_col}'. Available columns: {list(data.columns)}"
    )

# Convert all non-time columns to numeric so all weather variables can be used.
for col in data.columns:
    if col != time_col:
        data[col] = pd.to_numeric(data[col], errors="coerce")

# Start from all dataset columns except target source column and timestamp.
base_drop_cols = [c for c in [temp_col, time_col] if c in data.columns]
feature_df = data.drop(columns=base_drop_cols).copy()

# Remove columns that became entirely NaN after numeric conversion.
feature_df = feature_df.dropna(axis=1, how="all")

# Keep engineered temperature history features.
for h in [1, 2, 3, 6, 12]:
    feature_df[f"temp_lag_{h}"] = data[temp_col].shift(h)
    feature_df[f"dew_lag_{h}"] = data[dew_col].shift(h)
for h in [24, 48, 72, 96]:
    feature_df[f"temp_lag_{h}"] = data[temp_col].shift(h)
    feature_df[f"dew_lag_{h}"] = data[dew_col].shift(h)
    feature_df[f"temp_roll_mean_{h}"] = data[temp_col].shift(1).rolling(h).mean()
    feature_df[f"temp_roll_std_{h}"] = data[temp_col].shift(1).rolling(h).std()
    feature_df[f"temp_roll_min_{h}"] = data[temp_col].shift(1).rolling(h).min()
    feature_df[f"temp_roll_max_{h}"] = data[temp_col].shift(1).rolling(h).max()

if time_col in data.columns:
    feature_df["hour"] = data[time_col].dt.hour
    feature_df["dayofweek"] = data[time_col].dt.dayofweek
    feature_df["month"] = data[time_col].dt.month

feature_df["dew_lag_3"] = data[dew_col].shift(3)
feature_df["humidity_lag_3"] = data[humidity_col].shift(3)
feature_df["humidity_roll_mean_6"] = data[humidity_col].shift(1).rolling(6).mean()
feature_df["humidity_change"] = data[humidity_col] - data[humidity_col].shift(1)
feature_df["monthly_temp_mean"] = data[temp_col].shift(1).rolling(720).mean()
feature_df["pressure_gap"] = data["pressure_msl"] - data["surface_pressure"]

# Apply exclusion list to final model features.
selected_feature_cols = [col for col in feature_df.columns if col not in excluded_features]
print(f"Selected features for modeling: {selected_feature_cols}")
feature_df = feature_df[selected_feature_cols].copy()

target = data[temp_col].shift(-1)
model_data = feature_df.copy()
model_data["target"] = target
trainable = model_data.dropna().copy()
X = trainable.drop(columns="target")
y = trainable["target"]

if time_col not in data.columns:
    raise ValueError("Walk-forward validation requires a valid time column.")

# Build day-based walk-forward folds: 16 training days, 2 validation days, 2 test days.
trainable_times = data.loc[trainable.index, time_col].dt.normalize()
unique_days = pd.Index(trainable_times.dropna().unique()).sort_values()

train_days = 16
val_days = 2
test_days = 2
window_days = train_days + val_days + test_days

if len(unique_days) < window_days:
    raise ValueError(
        f"Need at least {window_days} unique days for walk-forward validation, got {len(unique_days)}."
    )

walk_forward_splits = []
for start_idx in range(len(unique_days) - window_days + 1):
    train_day_block = unique_days[start_idx:start_idx + train_days]
    val_day_block = unique_days[start_idx + train_days:start_idx + train_days + val_days]
    test_day_block = unique_days[start_idx + train_days + val_days:start_idx + window_days]

    train_mask = trainable_times.isin(train_day_block)
    val_mask = trainable_times.isin(val_day_block)
    test_mask = trainable_times.isin(test_day_block)

    if train_mask.sum() == 0 or val_mask.sum() == 0 or test_mask.sum() == 0:
        continue

    walk_forward_splits.append({
        "train_days": train_day_block,
        "val_days": val_day_block,
        "test_days": test_day_block,
        "train_idx": X.index[train_mask],
        "val_idx": X.index[val_mask],
        "test_idx": X.index[test_mask],
    })

if not walk_forward_splits:
    raise ValueError("No valid walk-forward splits could be created from the dataset.")

# Keep the latest fold as the active split so downstream cells continue to work unchanged.
active_split = walk_forward_splits[-1]
X_train = X.loc[active_split["train_idx"]]
X_val = X.loc[active_split["val_idx"]]
X_test = X.loc[active_split["test_idx"]]
y_train = y.loc[active_split["train_idx"]]
y_val = y.loc[active_split["val_idx"]]
y_test = y.loc[active_split["test_idx"]]

# Optional combined train+validation set for later experiments.
X_trainval_full = pd.concat([X_train, X_val])
y_trainval_full = pd.concat([y_train, y_val])

imputer = SimpleImputer(strategy="median")
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train), index=X_train.index, columns=X_train.columns
)
X_val_imputed = pd.DataFrame(
    imputer.transform(X_val), index=X_val.index, columns=X_val.columns
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test), index=X_test.index, columns=X_test.columns
)

next_step_features = feature_df.loc[[feature_df.index[-1]], X.columns].copy()
if not next_step_features.isna().all(axis=None):
    next_step_features_imputed = pd.DataFrame(
        imputer.transform(next_step_features),
        index=next_step_features.index,
        columns=next_step_features.columns,
    )
else:
    next_step_features_imputed = None

print(f"Walk-forward folds created: {len(walk_forward_splits)}")
print(
    f"Active fold -> train: {active_split['train_days'][0].date()} to {active_split['train_days'][-1].date()}, "
    f"val: {active_split['val_days'][0].date()} to {active_split['val_days'][-1].date()}, "
    f"test: {active_split['test_days'][0].date()} to {active_split['test_days'][-1].date()}"
)
print(f"Rows -> train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")

Selected features for modeling: ['relative_humidity_2m', 'dew_point_2m', 'rain', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_highh', 'wind_direction_10m', 'wind_gusts_10m', 'wind_direction_100m', 'wind_speed_100m', 'pressure_msl', 'surface_pressure', 'temp_lag_1', 'dew_lag_1', 'temp_lag_2', 'dew_lag_2', 'temp_lag_3', 'dew_lag_3', 'temp_lag_6', 'dew_lag_6', 'temp_lag_12', 'dew_lag_12', 'temp_lag_24', 'dew_lag_24', 'temp_roll_mean_24', 'temp_roll_std_24', 'temp_roll_min_24', 'temp_roll_max_24', 'temp_lag_48', 'dew_lag_48', 'temp_roll_mean_48', 'temp_roll_std_48', 'temp_roll_min_48', 'temp_roll_max_48', 'temp_lag_72', 'dew_lag_72', 'temp_roll_mean_72', 'temp_roll_std_72', 'temp_roll_min_72', 'temp_roll_max_72', 'temp_lag_96', 'dew_lag_96', 'temp_roll_mean_96', 'temp_roll_std_96', 'temp_roll_min_96', 'temp_roll_max_96', 'hour', 'month', 'humidity_lag_3', 'humidity_roll_mean_6', 'humidity_change', 'monthly_temp_mean', 'pressure_gap']
Walk-forward folds created: 346
Act

## Linear Regression Model

In [433]:
class MyLinearRegression:
    def __init__(self):
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X, y):
        X_np = np.asarray(X, dtype=float)
        y_np = np.asarray(y, dtype=float).reshape(-1, 1)

        # Add bias term
        X_b = np.c_[np.ones((X_np.shape[0], 1)), X_np]

        # Normal equation with pseudo-inverse for stability
        theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y_np

        self.intercept_ = float(theta[0, 0])
        self.coef_ = theta[1:, 0]
        return self

    def predict(self, X):
        X_np = np.asarray(X, dtype=float)
        return self.intercept_ + X_np @ self.coef_

import sklearn.linear_model

linear_model = sklearn.linear_model.LinearRegression()
linear_model.fit(X_train_imputed, y_train)

y_pred_val_lr = linear_model.predict(X_val_imputed)
y_pred_test_lr = linear_model.predict(X_test_imputed)

mae_val_lr = mean_absolute_error(y_val, y_pred_val_lr)
rmse_val_lr = mean_squared_error(y_val, y_pred_val_lr) ** 0.5
r2_val_lr = r2_score(y_val, y_pred_val_lr)

mae_test_lr = mean_absolute_error(y_test, y_pred_test_lr)
rmse_test_lr = mean_squared_error(y_test, y_pred_test_lr) ** 0.5
r2_test_lr = r2_score(y_test, y_pred_test_lr)

print("Linear Regression results")
print(f"Validation rows: {len(X_val)}")
print(f"Validation MAE:  {mae_val_lr:.3f}")
print(f"Validation RMSE: {rmse_val_lr:.3f}")
print(f"Validation R²:   {r2_val_lr:.3f}")
print(f"Test rows: {len(X_test)}")
print(f"Test MAE:  {mae_test_lr:.3f}")
print(f"Test RMSE: {rmse_test_lr:.3f}")
print(f"Test R²:   {r2_test_lr:.3f}")

forecast_results_lr = X_test.copy()
forecast_results_lr["actual_temperature_next_step"] = y_test.values
forecast_results_lr["predicted_temperature_next_step_lr"] = y_pred_test_lr

if next_step_features_imputed is not None:
    next_temperature_forecast_lr = float(linear_model.predict(next_step_features_imputed)[0])
    print(f"Next-step temperature forecast (Linear Regression): {next_temperature_forecast_lr:.3f}")
else:
    next_temperature_forecast_lr = None
    print("Next-step forecast could not be computed because the last row has missing features.")

linear_model_artifact_path = save_model_artifact(
    model_name="linear_regression",
    model=linear_model,
    imputer=imputer,
    feature_columns=X_train.columns,
    metadata={
        "validation_metrics": {"mae": mae_val_lr, "rmse": rmse_val_lr, "r2": r2_val_lr},
        "test_metrics": {"mae": mae_test_lr, "rmse": rmse_test_lr, "r2": r2_test_lr},
        "next_temperature_forecast": next_temperature_forecast_lr,
    },
)

forecast_results_lr.head()

Linear Regression results
Validation rows: 864
Validation MAE:  1.636
Validation RMSE: 2.058
Validation R²:   0.659
Test rows: 863
Test MAE:  1.695
Test RMSE: 2.175
Test R²:   0.598
Next-step temperature forecast (Linear Regression): 7.682
Saved linear_regression to c:\Users\Pedro Passos\Desktop\Weather-Prediction\level2\saved_models\linear_regression.joblib


,relative_humidity_2m,dew_point_2m,rain,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_highh,wind_direction_10m,wind_gusts_10m,wind_direction_100m,...,temp_roll_max_96,hour,month,humidity_lag_3,humidity_roll_mean_6,humidity_change,monthly_temp_mean,pressure_gap,actual_temperature_next_step,predicted_temperature_next_step_lr
157248,76,8.9,0.0,0,0,0,0,9,26.6,18,...,18.3,0,3,80.0,76.166667,2.0,13.003056,1.2,9.5,10.474169
157249,91,8.1,0.0,0,0,0,0,93,16.9,67,...,18.3,0,3,82.0,78.166667,15.0,13.004861,5.0,11.0,10.477329
157250,70,5.8,0.0,1,1,0,0,28,28.8,50,...,17.6,0,3,74.0,82.000000,-21.0,12.999722,55.0,11.9,10.585624
157251,81,8.7,0.0,2,0,2,0,119,8.6,83,...,17.6,0,3,76.0,78.833333,11.0,12.999444,1.1,11.8,10.317584
157252,75,7.5,0.0,6,0,0,5,25,28.1,36,...,17.4,0,3,91.0,79.000000,-6.0,13.004167,33.8,11.2,10.612712


In [434]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_imputed, y_train,
    eval_set=[(X_val_imputed, y_val)],
    verbose=False
)

y_pred_val_xgb = xgb_model.predict(X_val_imputed)
y_pred_test_xgb = xgb_model.predict(X_test_imputed)

mae_val_xgb = mean_absolute_error(y_val, y_pred_val_xgb)
rmse_val_xgb = mean_squared_error(y_val, y_pred_val_xgb) ** 0.5
r2_val_xgb = r2_score(y_val, y_pred_val_xgb)

mae_test_xgb = mean_absolute_error(y_test, y_pred_test_xgb)
rmse_test_xgb = mean_squared_error(y_test, y_pred_test_xgb) ** 0.5
r2_test_xgb = r2_score(y_test, y_pred_test_xgb)

print("XGBoost Regressor results")
print(f"Validation rows: {len(X_val)}")
print(f"Validation MAE:  {mae_val_xgb:.3f}")
print(f"Validation RMSE: {rmse_val_xgb:.3f}")
print(f"Validation R²:   {r2_val_xgb:.3f}")
print(f"Test rows: {len(X_test)}")
print(f"Test MAE:  {mae_test_xgb:.3f}")
print(f"Test RMSE: {rmse_test_xgb:.3f}")
print(f"Test R²:   {r2_test_xgb:.3f}")

forecast_results_xgb = X_test.copy()
forecast_results_xgb["actual_temperature_next_step"] = y_test.values
forecast_results_xgb["predicted_temperature_next_step_xgb"] = y_pred_test_xgb

if next_step_features_imputed is not None:
    next_temperature_forecast_xgb = float(xgb_model.predict(next_step_features_imputed)[0])
    print(f"Next-step temperature forecast (XGBoost): {next_temperature_forecast_xgb:.3f}")
else:
    next_temperature_forecast_xgb = None
    print("Next-step forecast could not be computed because the last row has missing features.")

xgb_model_artifact_path = save_model_artifact(
    model_name="xgboost_regressor",
    model=xgb_model,
    imputer=imputer,
    feature_columns=X_train.columns,
    metadata={
        "validation_metrics": {"mae": mae_val_xgb, "rmse": rmse_val_xgb, "r2": r2_val_xgb},
        "test_metrics": {"mae": mae_test_xgb, "rmse": rmse_test_xgb, "r2": r2_test_xgb},
        "next_temperature_forecast": next_temperature_forecast_xgb,
    },
)

forecast_results_xgb.head()

XGBoost Regressor results
Validation rows: 864
Validation MAE:  1.711
Validation RMSE: 2.134
Validation R²:   0.634
Test rows: 863
Test MAE:  1.736
Test RMSE: 2.229
Test R²:   0.578
Next-step temperature forecast (XGBoost): 5.531
Saved xgboost_regressor to c:\Users\Pedro Passos\Desktop\Weather-Prediction\level2\saved_models\xgboost_regressor.joblib


,relative_humidity_2m,dew_point_2m,rain,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_highh,wind_direction_10m,wind_gusts_10m,wind_direction_100m,...,temp_roll_max_96,hour,month,humidity_lag_3,humidity_roll_mean_6,humidity_change,monthly_temp_mean,pressure_gap,actual_temperature_next_step,predicted_temperature_next_step_xgb
157248,76,8.9,0.0,0,0,0,0,9,26.6,18,...,18.3,0,3,80.0,76.166667,2.0,13.003056,1.2,9.5,10.565798
157249,91,8.1,0.0,0,0,0,0,93,16.9,67,...,18.3,0,3,82.0,78.166667,15.0,13.004861,5.0,11.0,10.955619
157250,70,5.8,0.0,1,1,0,0,28,28.8,50,...,17.6,0,3,74.0,82.000000,-21.0,12.999722,55.0,11.9,11.130243
157251,81,8.7,0.0,2,0,2,0,119,8.6,83,...,17.6,0,3,76.0,78.833333,11.0,12.999444,1.1,11.8,11.321897
157252,75,7.5,0.0,6,0,0,5,25,28.1,36,...,17.4,0,3,91.0,79.000000,-6.0,13.004167,33.8,11.2,11.352406


In [435]:
from lightgbm import LGBMRegressor

lgbm_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    force_col_wise=True,
    verbosity=-1,
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(
    X_train_imputed,
    y_train,
    eval_set=[(X_val_imputed, y_val)],
    eval_metric="l2",
)

y_pred_val_lgbm = lgbm_model.predict(X_val_imputed)
y_pred_test_lgbm = lgbm_model.predict(X_test_imputed)

mae_val_lgbm = mean_absolute_error(y_val, y_pred_val_lgbm)
rmse_val_lgbm = mean_squared_error(y_val, y_pred_val_lgbm) ** 0.5
r2_val_lgbm = r2_score(y_val, y_pred_val_lgbm)

mae_test_lgbm = mean_absolute_error(y_test, y_pred_test_lgbm)
rmse_test_lgbm = mean_squared_error(y_test, y_pred_test_lgbm) ** 0.5
r2_test_lgbm = r2_score(y_test, y_pred_test_lgbm)

print("LightGBM Regressor results")
print(f"Validation rows: {len(X_val)}")
print(f"Validation MAE:  {mae_val_lgbm:.3f}")
print(f"Validation RMSE: {rmse_val_lgbm:.3f}")
print(f"Validation R²:   {r2_val_lgbm:.3f}")
print(f"Test rows: {len(X_test)}")
print(f"Test MAE:  {mae_test_lgbm:.3f}")
print(f"Test RMSE: {rmse_test_lgbm:.3f}")
print(f"Test R²:   {r2_test_lgbm:.3f}")

forecast_results_lgbm = X_test.copy()
forecast_results_lgbm["actual_temperature_next_step"] = y_test.values
forecast_results_lgbm["predicted_temperature_next_step_lgbm"] = y_pred_test_lgbm

if next_step_features_imputed is not None:
    next_temperature_forecast_lgbm = float(lgbm_model.predict(next_step_features_imputed)[0])
    print(f"Next-step temperature forecast (LightGBM): {next_temperature_forecast_lgbm:.3f}")
else:
    next_temperature_forecast_lgbm = None
    print("Next-step forecast could not be computed because the last row has missing features.")

lgbm_model_artifact_path = save_model_artifact(
    model_name="lightgbm_regressor",
    model=lgbm_model,
    imputer=imputer,
    feature_columns=X_train.columns,
    metadata={
        "validation_metrics": {"mae": mae_val_lgbm, "rmse": rmse_val_lgbm, "r2": r2_val_lgbm},
        "test_metrics": {"mae": mae_test_lgbm, "rmse": rmse_test_lgbm, "r2": r2_test_lgbm},
        "next_temperature_forecast": next_temperature_forecast_lgbm,
    },
)

forecast_results_lgbm.head()

LightGBM Regressor results
Validation rows: 864
Validation MAE:  1.708
Validation RMSE: 2.143
Validation R²:   0.631
Test rows: 863
Test MAE:  1.736
Test RMSE: 2.252
Test R²:   0.569
Next-step temperature forecast (LightGBM): 5.513
Saved lightgbm_regressor to c:\Users\Pedro Passos\Desktop\Weather-Prediction\level2\saved_models\lightgbm_regressor.joblib


,relative_humidity_2m,dew_point_2m,rain,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_highh,wind_direction_10m,wind_gusts_10m,wind_direction_100m,...,temp_roll_max_96,hour,month,humidity_lag_3,humidity_roll_mean_6,humidity_change,monthly_temp_mean,pressure_gap,actual_temperature_next_step,predicted_temperature_next_step_lgbm
157248,76,8.9,0.0,0,0,0,0,9,26.6,18,...,18.3,0,3,80.0,76.166667,2.0,13.003056,1.2,9.5,10.639200
157249,91,8.1,0.0,0,0,0,0,93,16.9,67,...,18.3,0,3,82.0,78.166667,15.0,13.004861,5.0,11.0,10.379093
157250,70,5.8,0.0,1,1,0,0,28,28.8,50,...,17.6,0,3,74.0,82.000000,-21.0,12.999722,55.0,11.9,12.323117
157251,81,8.7,0.0,2,0,2,0,119,8.6,83,...,17.6,0,3,76.0,78.833333,11.0,12.999444,1.1,11.8,11.919725
157252,75,7.5,0.0,6,0,0,5,25,28.1,36,...,17.4,0,3,91.0,79.000000,-6.0,13.004167,33.8,11.2,11.876082


In [436]:
# Average the test predictions from Random Forest, XGBoost, and LightGBM.
# Use index alignment because the RF results were created in a different cell/setup.

rf_test = forecast_results[
    ["actual_temperature_next_step", "predicted_temperature_next_step"]
].rename(columns={"predicted_temperature_next_step": "pred_rf"})

xgb_test = forecast_results_xgb[
    ["actual_temperature_next_step", "predicted_temperature_next_step_xgb"]
].rename(columns={"predicted_temperature_next_step_xgb": "pred_xgb"})

lgbm_test = forecast_results_lgbm[
    ["actual_temperature_next_step", "predicted_temperature_next_step_lgbm"]
].rename(columns={"predicted_temperature_next_step_lgbm": "pred_lgbm"})

forecast_results_vote = (
    rf_test[["actual_temperature_next_step", "pred_rf"]]
    .join(xgb_test[["pred_xgb"]], how="inner")
    .join(lgbm_test[["pred_lgbm"]], how="inner")
    .copy()
)

forecast_results_vote["predicted_temperature_next_step_vote"] = forecast_results_vote[
    ["pred_rf", "pred_xgb", "pred_lgbm"]
].mean(axis=1)

mae_test_vote = mean_absolute_error(
    forecast_results_vote["actual_temperature_next_step"],
    forecast_results_vote["predicted_temperature_next_step_vote"]
)
rmse_test_vote = mean_squared_error(
    forecast_results_vote["actual_temperature_next_step"],
    forecast_results_vote["predicted_temperature_next_step_vote"]
) ** 0.5
r2_test_vote = r2_score(
    forecast_results_vote["actual_temperature_next_step"],
    forecast_results_vote["predicted_temperature_next_step_vote"]
)

print("Voting/Averaging Ensemble results")
print(f"Test rows used: {len(forecast_results_vote)}")
print(f"Test MAE:  {mae_test_vote:.3f}")
print(f"Test RMSE: {rmse_test_vote:.3f}")
print(f"Test R²:   {r2_test_vote:.3f}")

next_temperature_forecast_vote = np.mean([
    next_temperature_forecast,
    next_temperature_forecast_xgb,
    next_temperature_forecast_lgbm
])
print(f"Next-step temperature forecast (Voting/Average): {next_temperature_forecast_vote:.3f}")

forecast_results_vote.head()

Voting/Averaging Ensemble results
Test rows used: 863
Test MAE:  1.662
Test RMSE: 2.156
Test R²:   0.605
Next-step temperature forecast (Voting/Average): 6.119


,actual_temperature_next_step,pred_rf,pred_xgb,pred_lgbm,predicted_temperature_next_step_vote
157248,9.5,11.548255,10.565798,10.639200,10.917751
157249,11.0,11.801986,10.955619,10.379093,11.045566
157250,11.9,11.725479,11.130243,12.323117,11.726280
157251,11.8,11.639798,11.321897,11.919725,11.627140
157252,11.2,11.975780,11.352406,11.876082,11.734756
